## Basic RAG Pipeline Example

In [1]:
!pip install -U transformers sentence-transformers faiss-cpu ipywidgets

   ---------------------------------------- 0.0/12.8 MB ? eta -:--:--
   - -------------------------------------- 0.5/12.8 MB 3.6 MB/s eta 0:00:04
   ------------ --------------------------- 3.9/12.8 MB 12.7 MB/s eta 0:00:01
   -------------------------- ------------- 8.7/12.8 MB 17.1 MB/s eta 0:00:01
   ---------------------------------------  12.6/12.8 MB 18.3 MB/s eta 0:00:01
   ---------------------------------------- 12.8/12.8 MB 16.5 MB/s  0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 5.19.0
    Uninstalling transformers-5.19.0:
      Successfully uninstalled transformers-5.19.0


### RAW Knowledge Base

In [2]:
DOCUMENTS = [
    "The Eiffel Tower is located in Paris.",
    "The Great Wall of China is visible from space.",
    "The Amazon rainforest produces 20% of the world's oxygen.",
    "A new element, Bozanium, was discovered in 2025. It's atomic weight is higher than iron.",
    "Bozanium is a rare earth element found mainly in the country of Bozania.",
    "Research shows that to make a fire-proof plastic sheet, trace amounts of Bozanium are needed.",
]

In [3]:
query = "What can you tell me about Bozanium?"

## Initialize HuggingFace Models

We require two models here, one for embedding and another for the LLM itself. For each we require the tokenizer which goes with it.

In [4]:
HF_LLM_MODEL_ID = "unsloth/Llama-3.2-1B-Instruct"

HF_EMBEDDING_MODEL_ID = "all-MiniLM-L6-v2"   # "all-mpnet-base-v2"

DEVICE = "cpu"   # "cpu"

In [6]:
import os

import numpy as np
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
from sentence_transformers import SentenceTransformer

In [7]:
LLM_TOKENIZER = AutoTokenizer.from_pretrained(HF_LLM_MODEL_ID)
LLM_MODEL = AutoModelForCausalLM.from_pretrained(HF_LLM_MODEL_ID, dtype=torch.bfloat16, device_map=DEVICE)

EMBEDDING_MODEL = SentenceTransformer(model_name_or_path=HF_EMBEDDING_MODEL_ID, device=DEVICE).to(DEVICE)

c:\Users\Efrat\Downloads\KodCode-AI-Augmented-Software-Development\.venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Efrat\.cache\huggingface\hub. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


model.safetensors: reconstructing file:   0%|          |  0.00B / 2.47GB            

model.safetensors: downloading bytes:           |  0.00B            

c:\Users\Efrat\Downloads\KodCode-AI-Augmented-Software-Development\.venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Efrat\.cache\huggingface\hub\models--unsloth--Llama-3.2-1B-Instruct. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

## Embedding

In [8]:
normalized_doc_embeddings = EMBEDDING_MODEL.encode(DOCUMENTS)

In [9]:
print(normalized_doc_embeddings)

[[ 0.06605348  0.03884843  0.01661565 ...  0.03093833  0.07990991
   0.05157553]
 [ 0.04052193  0.07120398  0.04857148 ...  0.01960543 -0.09164688
   0.07383972]
 [ 0.06734324  0.04965374 -0.017277   ...  0.00083842  0.04014445
  -0.00207049]
 [-0.09929054  0.02193136 -0.03951164 ... -0.11316758  0.01971994
   0.02981951]
 [-0.04054     0.01736863 -0.03872806 ... -0.00845237  0.03973871
   0.03547049]
 [-0.12465699  0.07095066 -0.01322104 ...  0.01488636  0.06807771
   0.05093715]]


In [11]:
normalized_doc_embeddings.shape

(6, 384)

## Indexing

We'll use an in-memory vector database from Meta - *FAISS* (what does it sound to you like 😊?)

In [15]:
import faiss

The `sentence_transformers` package takes care of pooling and normalization for us.

In [16]:
# https://www.restack.io/p/vector-database-answer-faiss-cosine-similarity-cat-ai
# IndexFlatIP - Inner Product - equivalent to cosine-similarity, given the embeddings are normalized
FAISS_INDEX = faiss.IndexFlatIP(normalized_doc_embeddings.shape[-1])

In [17]:
FAISS_INDEX.add(normalized_doc_embeddings)

In [19]:
# Persisting. Note that the documents are not persisted alongside the embeddings.

faiss.write_index(FAISS_INDEX, "doc-embeddings.index")
# FAISS_INDEX = faiss.read_index(FAISS_INDEX,"doc-embeddings.index")

## Retriever

Returns the most relevant documents based on the similarity of their embeddings to the query embedding.

In [20]:
query_embedding = EMBEDDING_MODEL.encode([query])

In [21]:
query_embedding.shape

(1, 384)

In [22]:
faiss_similarity_scores, faiss_indices = FAISS_INDEX.search(query_embedding, 5)

In [23]:
faiss_similarity_scores

array([[0.6995909 , 0.5752559 , 0.4661682 , 0.08350903, 0.04974772]],
      dtype=float32)

In [24]:
faiss_indices

array([[4, 3, 5, 1, 2]])

In [25]:
print("Query:", query)
[f"[{score:.3f}] {DOCUMENTS[i]}" for i, score in zip(faiss_indices[0], faiss_similarity_scores[0])]

Query: What can you tell me about Bozanium?


['[0.700] Bozanium is a rare earth element found mainly in the country of Bozania.',
 "[0.575] A new element, Bozanium, was discovered in 2025. It's atomic weight is higher than iron.",
 '[0.466] Research shows that to make a fire-proof plastic sheet, trace amounts of Bozanium are needed.',
 '[0.084] The Great Wall of China is visible from space.',
 "[0.050] The Amazon rainforest produces 20% of the world's oxygen."]

In [30]:
def retrieve_documents(query_text, top_k=3, similarity_cutoff=0.4):
    
    query_embedding = EMBEDDING_MODEL.encode([query_text])
    faiss_similarity_scores, faiss_indices = FAISS_INDEX.search(query_embedding, top_k)
    
    return [DOCUMENTS[i] for rag_doc_i, i in enumerate(faiss_indices[0]) if faiss_similarity_scores[0][rag_doc_i] >= similarity_cutoff]

In [32]:
retrieve_documents(query)

['Bozanium is a rare earth element found mainly in the country of Bozania.',
 "A new element, Bozanium, was discovered in 2025. It's atomic weight is higher than iron.",
 'Research shows that to make a fire-proof plastic sheet, trace amounts of Bozanium are needed.']

## Generator

In [33]:
pipeline?

Signature:
pipeline(
    task: 'str | None' = None,
    model: 'str | PreTrainedModel | None' = None,
    config: 'str | PreTrainedConfig | None' = None,
    tokenizer: 'str | PreTrainedTokenizer | PreTrainedTokenizerFast | None' = None,
    feature_extractor: 'str | FeatureExtractionMixin | None' = None,
    image_processor: 'str | BaseImageProcessor | None' = None,
    video_processor: 'str | BaseVideoProcessor | None' = None,
    processor: 'str | ProcessorMixin | None' = None,
    revision: 'str | None' = None,
    use_fast: 'bool' = True,
    token: 'str | bool | None' = None,
    device: 'int | str | torch.device | None' = None,
    device_map: 'str | dict[str, int | str] | None' = None,
    dtype: 'str | torch.dtype | None' = 'auto',
    trust_remote_code: 'bool | None' = None,
    model_kwargs: 'dict[str, Any] | None' = None,
    pipeline_class: 'Any | None' = None,
    **kwargs: 'Any',
) -> 'Pipeline'
Docstring:
Utility factory method to build a [`Pipeline`].

A pipeline consi

In [34]:
GENERATOR = pipeline(task="text-generation", model=LLM_MODEL, tokenizer=LLM_TOKENIZER)

**Without** contextual augmentation:

In [35]:
response = GENERATOR(query, max_new_tokens=300, return_full_text=False, temperature=0.01)
print(response[0]["generated_text"])

[transformers] Both `max_new_tokens` (=300) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


 Bozanium is a popular restaurant chain that specializes in BBQ and American comfort food. They have a wide variety of menu items, including meats, sides, and desserts. Bozanium is known for its casual, welcoming atmosphere and its commitment to using high-quality ingredients.

Bozanium has several locations throughout the United States, including in major cities like New York, Los Angeles, and Chicago. They also have a strong online presence, with a website and social media accounts that allow customers to order food online and track their orders.

One of the unique features of Bozanium is its "Boz" mascot, a friendly, anthropomorphic pig named Boz. Boz is a beloved character who has become a symbol of the brand and is often featured on menus, merchandise, and promotional materials.

Overall, Bozanium is a popular destination for those looking for a casual, delicious meal in a welcoming atmosphere. Whether you're in the mood for BBQ, comfort food, or something a little more adventurou

Well, its all hallucinations...

In [36]:
def generate_answer(query_text, rag_documents: list[str]) -> str:

    context = f"""
    Context information is provided to augment your knowledge. 
    Context: {os.linesep.join(rag_documents)}
    """ if rag_documents else ""

    prompt = f"""
    You are a smart AI assistant. Only reply with your final answer.
    {context}\n
    Question: {query_text}\n
    Answer:"""
    
    response = GENERATOR(prompt, max_new_tokens=300, return_full_text=False, temperature=0.01)

    return response[0]["generated_text"]

### RAG Pipeline

In [37]:
from IPython.display import display, Markdown

In [38]:
def rag_pipeline(query_text, use_context=True, display_answer=True):

    rag_documents = retrieve_documents(query_text) if use_context else None
    answer = generate_answer(query_text, rag_documents)

    if display_answer:
        print("Query:", query_text, f"" if use_context else "// NO CONTEXT")
        display(Markdown(answer))
    else:
        return answer

## Testing

In [39]:
query_1 = "What can you tell me about Bozanium?"

In [40]:
rag_pipeline(query_1, use_context=False)

[transformers] Both `max_new_tokens` (=300) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Query: What can you tell me about Bozanium? // NO CONTEXT


 Bozanium is a restaurant chain that specializes in gourmet grilled cheese sandwiches.

In [41]:
retrieve_documents(query_1)

['Bozanium is a rare earth element found mainly in the country of Bozania.',
 "A new element, Bozanium, was discovered in 2025. It's atomic weight is higher than iron.",
 'Research shows that to make a fire-proof plastic sheet, trace amounts of Bozanium are needed.']

In [42]:
rag_pipeline(query_1, use_context=True)

[transformers] Both `max_new_tokens` (=300) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Query: What can you tell me about Bozanium? 


 Bozanium is a rare earth element discovered in 2025, with an atomic weight higher than iron. It's needed to make fire-proof plastic sheets.

In [43]:
query_2 = "What could help make firemen suits made of plastic more fireproof?"

In [44]:
rag_pipeline(query_2, use_context=False)

[transformers] Both `max_new_tokens` (=300) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Query: What could help make firemen suits made of plastic more fireproof? // NO CONTEXT


 Using flame retardant materials and advanced textile technologies, such as those used in fire-resistant fabrics, could help make firemen's suits more fireproof.

In [45]:
retrieve_documents(query_2)

['Research shows that to make a fire-proof plastic sheet, trace amounts of Bozanium are needed.']

In [46]:
rag_pipeline(query_2, use_context=True)

[transformers] Both `max_new_tokens` (=300) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Query: What could help make firemen suits made of plastic more fireproof? 


 Adding trace amounts of Bozanium to the plastic could help make firemen's suits more fireproof.

In [47]:
query_3 = "Have new elements been discovered lately?"

In [48]:
rag_pipeline(query_3, use_context=False)

[transformers] Both `max_new_tokens` (=300) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


KeyboardInterrupt: 

In [49]:
retrieve_documents(query_3)

["A new element, Bozanium, was discovered in 2025. It's atomic weight is higher than iron."]

In [50]:
rag_pipeline(query_3, use_context=True)

[transformers] Both `max_new_tokens` (=300) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Query: Have new elements been discovered lately? 


 Yes, several new elements have been discovered in recent years, including Bozanium, which was discovered in 2025.

In [ ]:
query_4 = "What is the importance of rainforests?"

In [ ]:
rag_pipeline(query_4, use_context=False)

In [ ]:
retrieve_documents(query_4)

In [ ]:
rag_pipeline(query_4, use_context=True)

<hr/>